# Hindi-HDTB Data Exploration

## Research project
Partial Freezing of Multilingual Language Models for POS Tagging

## Objective
Understand the Hindi Universal Dependencies treebank before preparing
the data for a token-classification model.

## Dataset
Universal Dependencies Hindi-HDTB

## Initial research questions
1. How are sentences represented in CoNLL-U?
2. How many sentences and tokens are in the training split?
3. Which UPOS labels appear in the dataset?

In [1]:
import pandas as pd
import random
import json
print("Pandas version:", pd.__version__)

Pandas version: 3.0.6


In [2]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.conllu_utils import read_conllu_sentences, count_conllu

In [3]:
data_path = Path("../data/hi_hdtb-ud-train.conllu")

print("Dataset exists:", data_path.exists())
print("Dataset path:", data_path.resolve())

Dataset exists: True
Dataset path: /Users/subekxya/Documents/MLP/data/hi_hdtb-ud-train.conllu


The dataset file exists, and its absolute path was verified successfully.

### Inspecting the Dataset

In [4]:
with data_path.open("r", encoding='utf-8') as file:
    for i in range(30):
        line = file.readline()
        print(line, end="")

# sent_id = train-s1
# text = यह एशिया की सबसे बड़ी मस्जिदों में से एक है ।
# translit = yaha eśiyā kī sabase baṛī masjidoṁ meṁ se eka hai .
1	यह	यह	DET	DEM	Case=Nom|Number=Sing|Person=3|PronType=Dem	2	det	_	ChunkId=NP|ChunkType=child|LTranslit=yaha|Translit=yaha
2	एशिया	एशिया	PROPN	NNP	Case=Acc|Gender=Masc|Number=Sing|Person=3	6	nmod	_	ChunkId=NP|ChunkType=head|LTranslit=eśiyā|Tam=0|Translit=eśiyā|Vib=0_का
3	की	का	ADP	PSP	AdpType=Post|Case=Acc|Gender=Fem|Number=Plur	2	case	_	ChunkId=NP|ChunkType=child|LTranslit=kā|Translit=kī
4	सबसे	सबसे	ADV	INTF	AdvType=Deg	5	advmod	_	AltTag=avy-ADV|ChunkId=NP2|ChunkType=child|LTranslit=sabase|Translit=sabase
5	बड़ी	बड़ा	ADJ	JJ	Case=Acc|Gender=Fem|Number=Plur	6	amod	_	ChunkId=NP2|ChunkType=child|LTranslit=baṛā|Translit=baṛī
6	मस्जिदों	मस्जिद	NOUN	NN	Case=Acc|Gender=Fem|Number=Plur|Person=3	9	nmod	_	ChunkId=NP2|ChunkType=head|LTranslit=masjida|Tam=0|Translit=masjidoṁ|Vib=0_में_से
7	में	में	ADP	PSP	AdpType=Post	6	case	_	ChunkId=NP2|ChunkType=child|LTra

The first 30 lines display the Hindi UD-HDTB dataset in CoNLL-U format, including sentence metadata, Hindi text, transliteration, and token-level annotations.

Each token row contains information such as the word, lemma, part-of-speech tag, grammatical features, and dependency relation.

### Loading the Dataset into a DataFrame

In [5]:
column_names = ["ID", "FORM", "LEMMA", "UPOS", "XPOS", "FEATS", "HEAD", "DEPREL", "DEPS", "MISC"]

rows = []

with data_path.open("r", encoding='utf-8') as file:
    for line in file:
        line = line.strip()

        if not line or line.startswith("#"):
            continue

        columns = line.split("\t")

        if len(columns) == 10:
            rows.append(columns)

df = pd.DataFrame(rows, columns=column_names)

df.head()

,ID,FORM,LEMMA,UPOS,XPOS,FEATS,HEAD,DEPREL,DEPS,MISC
0,1,यह,यह,DET,DEM,Case=Nom|Number=Sing|Person=3|PronType=Dem,2,det,_,ChunkId=NP|ChunkType=child|LTranslit=yaha|Tran...
1,2,एशिया,एशिया,PROPN,NNP,Case=Acc|Gender=Masc|Number=Sing|Person=3,6,nmod,_,ChunkId=NP|ChunkType=head|LTranslit=eśiyā|Tam=...
2,3,की,का,ADP,PSP,AdpType=Post|Case=Acc|Gender=Fem|Number=Plur,2,case,_,ChunkId=NP|ChunkType=child|LTranslit=kā|Transl...
3,4,सबसे,सबसे,ADV,INTF,AdvType=Deg,5,advmod,_,AltTag=avy-ADV|ChunkId=NP2|ChunkType=child|LTr...
4,5,बड़ी,बड़ा,ADJ,JJ,Case=Acc|Gender=Fem|Number=Plur,6,amod,_,ChunkId=NP2|ChunkType=child|LTranslit=baṛā|Tra...


A DataFrame displaying the first five token rows with 10 columns containing the CoNLL-U annotations.

The code skips empty lines and comments, keeps valid 10-column token rows, and organizes them into a pandas DataFrame for further analysis.

### UPOS Label Distribution

In [6]:
print("Number of token rows:", len(df))

print("UPOS label counts:", df["UPOS"].value_counts())

print("Number of unique UPOS labels:", df["UPOS"].nunique())

upos_percentage = df["UPOS"].value_counts(normalize=True) * 100

upos_percentage.round(2)

Number of token rows: 281057
UPOS label counts: UPOS
NOUN     62191
ADP      58463
PROPN    34288
VERB     29008
AUX      18947
PUNCT    18668
ADJ      17272
PRON     11857
DET       6081
PART      5610
SCONJ     5389
NUM       5332
CCONJ     5110
ADV       2711
X          127
INTJ         3
Name: count, dtype: int64
Number of unique UPOS labels: 16


UPOS
NOUN     22.13
ADP      20.80
PROPN    12.20
VERB     10.32
AUX       6.74
PUNCT     6.64
ADJ       6.15
PRON      4.22
DET       2.16
PART      2.00
SCONJ     1.92
NUM       1.90
CCONJ     1.82
ADV       0.96
X         0.05
INTJ      0.00
Name: proportion, dtype: float64

The dataset contains 281,057 token rows and 16 unique UPOS labels. NOUN (22.13%) and ADP (20.80%) are the most frequent, while INTJ occurs least frequently (3 tokens).

The label distribution shows class imbalance, which should be considered when evaluating the UPOS tagging model.

### Sentence-Level Dataset Inspection

In [7]:
sentences = read_conllu_sentences(data_path)

print("Number of sentences:", len(sentences))
print("Number of word tokens:", sum(len(sentence) for sentence in sentences))
print("Token in first sentence:", len(sentences[0]))
print("First sentence:", sentences[0])

Number of sentences: 13306
Number of word tokens: 281057
Token in first sentence: 11
First sentence: [{'ID': 1, 'FORM': 'यह', 'UPOS': 'DET'}, {'ID': 2, 'FORM': 'एशिया', 'UPOS': 'PROPN'}, {'ID': 3, 'FORM': 'की', 'UPOS': 'ADP'}, {'ID': 4, 'FORM': 'सबसे', 'UPOS': 'ADV'}, {'ID': 5, 'FORM': 'बड़ी', 'UPOS': 'ADJ'}, {'ID': 6, 'FORM': 'मस्जिदों', 'UPOS': 'NOUN'}, {'ID': 7, 'FORM': 'में', 'UPOS': 'ADP'}, {'ID': 8, 'FORM': 'से', 'UPOS': 'ADP'}, {'ID': 9, 'FORM': 'एक', 'UPOS': 'NUM'}, {'ID': 10, 'FORM': 'है', 'UPOS': 'AUX'}, {'ID': 11, 'FORM': '।', 'UPOS': 'PUNCT'}]


The dataset contains 13,306 sentences and 281,057 word tokens. The first sentence has 11 tokens, each represented by its ID, Hindi word (FORM), and UPOS tag.

The data is organized by sentence, ready for token-level sequence labeling.

### Data Quality and Sentence Length

In [8]:
#Check the missing or invalid values
print("Missing FORM values:", sum(token["FORM"] == "" for s in sentences for token in s))
print("Missing UPOS values:", sum(token["UPOS"] == "_" for s in sentences for token in s))

#Check sentence lengths
sentence_lengths = [len(sentence) for sentence in sentences]

print(f"Shortest sentences: {min(sentence_lengths)} tokens.")
print(f"Longest tokens: {max(sentence_lengths)} tokens.")
print(f"Average sentence length: {sum(sentence_lengths)/len(sentence_lengths)}")

Missing FORM values: 0
Missing UPOS values: 0
Shortest sentences: 1 tokens.
Longest tokens: 116 tokens.
Average sentence length: 21.122576281376823


No missing FORM or UPOS values were found. Sentence lengths range from 1 to 116 tokens, with an average of approximately 21.12 tokens.

### Dataset Split Statistics

In [9]:
data_dir = project_root / "data"

file_names = ["hi_hdtb-ud-train.conllu", "hi_hdtb-ud-dev.conllu", "hi_hdtb-ud-test.conllu"]

for file_name in file_names:
    result = count_conllu(data_dir / file_name)

    print(f"{file_name}")
    print(f"Sentences: {result['sentences']:,}")
    print(f"Word tokens: {result['word_tokens']:,}")


hi_hdtb-ud-train.conllu
Sentences: 13,306
Word tokens: 281,057
hi_hdtb-ud-dev.conllu
Sentences: 1,659
Word tokens: 35,217
hi_hdtb-ud-test.conllu
Sentences: 1,684
Word tokens: 35,430


The dataset is divided into three splits: the training set contains 13,306 sentences and 281,057 word tokens, the development set contains 1,659 sentences and 35,217 word tokens, and the test set contains 1,684 sentences and 35,430 word tokens. In total, the dataset contains 16,649 sentences and 351,704 word tokens.

### Load Training Data

In [10]:
train_path = data_dir / "hi_hdtb-ud-train.conllu"
train_sentences = read_conllu_sentences(train_path)

print(f"Number of training sentences: {len(train_sentences)}")

Number of training sentences: 13306


The training dataset was loaded successfully, containing 13,306 sentences.